# 线性回归从零实现

## 导包

In [1]:
import random
import torch
print(torch.__version__)

1.12.0+cu113


## 构造数据集
使用线性模型参数$\mathbf{w} = [2, -3.4]^\top$、$b = 4.2$
和噪声项$\epsilon$生成数据集及其标签:
$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \mathbf\epsilon.$$

In [2]:
def synthetic_data(w, b, num_examples):  
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples,len(w))) #X的列和w的长度相同
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 1, y.size()) #噪声
    return X, y.reshape((-1, 1)) #返回x和y,这里y是二维的

In [3]:
#测试
test1 = torch.normal(0, 1, (1,3))
test1

tensor([[1.5254, 1.4336, 1.1280]])

形状验证

In [4]:
num_examples = 100
w = torch.tensor([2,-3.4], dtype=torch.float32)  
b = torch.tensor(4.2, dtype=torch.float32)

features, labels = synthetic_data(w, b, num_examples)  #w被广播为(1,3)然后与X相乘
print(f"feature[0]: {features[0]}", features.shape, "\n", f"labels[0]: {labels[0]}", labels.shape)


feature[0]: tensor([-0.5928, -0.5893]) torch.Size([100, 2]) 
 labels[0]: tensor([5.3726]) torch.Size([100, 1])


## 读取数据集
该函数能打乱数据集中的样本并以小批量方式获取数据




In [5]:
def data_iter(batch_size, features, labels):
    num_features = len(features)
    index_features = list(range(num_features))
    # 打乱样本，随机读取
    random.shuffle(index_features)
    #小批次读取
    for i in range(0, num_features, batch_size):
        batch_features = torch.tensor(index_features[i:min(i + batch_size, num_features)]) #索引不要搞错
        yield features[batch_features], labels[batch_features]



采用小批量，因此函数执行一次就采样一小批样本

In [37]:
batch_size = 10

for X, y in data_iter(batch_size, features, labels):
    print(X.shape, '\n', y.shape)
    break #取第一个batch

    

torch.Size([10, 2]) 
 torch.Size([10, 1])


## 初始化模型参数，定义模型、损失函数以及优化算法

通过从均值为0、标准差为0.01的正态分布中采样随机数来初始化权重，
并将偏置初始化为0。

优化算法是sgd：小批量随机梯度下降更新参数

In [7]:
#初始化模型参数
W = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

#模型
def linreg(X, W, b):
    return torch.matmul(X, W) + b

#损失函数
def squared_loss(y_hat, y):
    loss = (y_hat - y.reshape(y_hat.shape)) **2 / 2   #把数据集的标签y reshape成y_hat的形状，是一份保险
    return loss

#优化算法
def sgd(params, lr, batch_size):
    with torch.no_grad():  #更新参数值时要禁用梯度计算
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()  #梯度清零

## 训练
先算loss、再反向传播求梯度，最后用sgd更新参数，并打印epoch和loss

In [8]:
lr = 0.1
batch_size = 10

for epoch in range(5):
    for X, y in data_iter(batch_size, features, labels):
        y_hat = linreg(X, W, b)
        loss = squared_loss(y_hat, y)   #loss是二维，形状是[batch,1]
        loss.sum().backward()   #必须sum成标量才能进行反向传播
        sgd([W,b], lr, batch_size) # 使用参数的梯度更新参数
    with torch.no_grad(): 
        train_loss = squared_loss(linreg(features, W, b), labels)  #注意这里用的是数据集的features和labels，唯一变的是W
        print(f'epoch: {epoch} ', f'train_loss: {train_loss.mean()}')  #这里用mean打印平均损失，不然放不下

epoch: 0  train_loss: 2.779813766479492
epoch: 1  train_loss: 0.9191979765892029
epoch: 2  train_loss: 0.5762869119644165
epoch: 3  train_loss: 0.5149476528167725
epoch: 4  train_loss: 0.5015868544578552


# 线性回归的简洁实现

In [95]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

## 生成数据集
用dl包的synthetic_data函数

In [96]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)
features.shape

torch.Size([1000, 2])

## 读取数据集
调用TensorDataset创建dataset(TensorDataset只要传一个参数，很快，用Dataset稍微麻烦了点)，返回Dataloader

In [97]:
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)  #解包数据集
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [98]:
dataloader = load_array((features, labels), 100, True)  #传入元组然后解包
x,y = next(iter(dataloader))
print(x.shape, y.shape)

torch.Size([100, 2]) torch.Size([100, 1])


## 定义模型

使用nn.Sequential容器，用现成的框架创建神经网络层，代码更简洁方便

In [99]:
from torch import nn 
network = nn.Sequential(nn.Linear(in_features=2, out_features=1))

## 初始化模型参数、定义损失函数MSE、定义优化算法SGD
初始化权重和偏置，权重w的均值和标准差为0和0.01，注意使用`weight.data`和`bias.data`方法访问参数。

In [ ]:
# 初始化模型参数
network[0].weight.data.normal_(0, 0.01)
network[0].bias.data.fill_(0)

# 损失函数
lf = nn.MSELoss()

# 优化算法
# lr = 0.05  #再增加就可能会导致第7轮的loss反向增长
lr = 0.03  #这个最合适
optim = torch.optim.SGD(network.parameters(), lr)

## 训练

In [101]:
num_epoch = 10

for epoch in range(num_epoch):
    for x,y in dataloader:
        loss = lf(network(x),y)
        #实际上是在操纵传给优化器对象
        optim.zero_grad()
        loss.backward()
        optim.step()
    train_loss = lf(network(features), labels) #打印损失的时候要用数据集的标签
    print(f"epoch: {epoch}", f"train_loss: {train_loss.mean()}")

epoch: 0 train_loss: 9.663176536560059
epoch: 1 train_loss: 2.867663860321045
epoch: 2 train_loss: 0.8539555668830872
epoch: 3 train_loss: 0.2551039457321167
epoch: 4 train_loss: 0.07643988728523254
epoch: 5 train_loss: 0.023023461923003197
epoch: 6 train_loss: 0.006992070935666561
epoch: 7 train_loss: 0.00217631459236145
epoch: 8 train_loss: 0.0007252190262079239
epoch: 9 train_loss: 0.00028595211915671825


In [ ]:
w1 = network[0].weight.data
b1 = network[0].bias.data
print(f" w的误差: {w - w1.reshape(w.shape)}", "\n", f"b的误差: {b - b1}")  # 维度对齐(保险)


 w的误差: tensor([[ 0.0029, -0.0080]]) 
 b的误差: tensor([-0.0452], grad_fn=<SubBackward0>)
